In [ ]:
import h5py
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.neighbors import NearestNeighbors
from torch.utils.data import DataLoader, Dataset

In [ ]:
sns.set_theme()
%matplotlib inline

visualize = True

In [ ]:
filename = "/scratch/gpfs/EKOLEMEN/hackathon/170000.h5"

In [ ]:
with h5py.File(filename, "r") as f:
    co2_data = f["co2_phase"]
    print(co2_data.keys())
    print(co2_data.attrs.keys())
    co2_channels = co2_data.attrs["channel_ids"]
    co2_samples = co2_data["data"][()]
    co2_time_ms = co2_data["time"][()]

In [ ]:
with h5py.File(filename, "r") as f:
    mse_data = f["mse"]
    print(mse_data.attrs.keys())
    print(mse_data.attrs["channel_ids"])
    mse_channels = mse_data.attrs["channel_ids"]
    mse_samples = mse_data["data"][()]
    mse_time_ms = mse_data["time"][()]

In [ ]:
if visualize:
    fig, axs = plt.subplots()

    for k, label in enumerate(["01", "02", "03", "04"]):
        sns.lineplot(x=mse_time_ms[:, k], y=mse_samples[:, k], label=label)
    axs.set_xlabel("Time (ms)")
    axs.set_ylabel("MSE")
    axs.set_xlim((1800, 1850))

In [ ]:
if visualize:
    fig, axs = plt.subplots()

    for k, label in enumerate(["01", "02", "03", "04"]):
        sns.lineplot(x=mse_time_ms[:, k], y=mse_samples[:, k], label=label)
    axs.set_xlabel("Time (ms)")
    axs.set_ylabel("MSE")
    axs.set_xlim((1800, 1850))

In [ ]:
if visualize:
    fig, axs = plt.subplots()

    axs.vlines(x=mse_time_ms[:, 0], ymin=-1600, ymax=800, linewidths=(0.1,))
    for k, label in enumerate(["r0", "v1", "v2", "v3"]):
        sns.lineplot(x=co2_time_ms[:, k], y=co2_samples[:, k], label=label)
    axs.set_xlabel("Time (ms)")
    axs.set_ylabel("CO2")
    axs.set_xlim((1800, 1850))

In [ ]:
nbrs = NearestNeighbors(n_neighbors=1).fit(co2_time_ms[:, 0].reshape(-1, 1))
distances, indices = nbrs.kneighbors(mse_time_ms[:, 0].reshape(-1, 1))

In [ ]:
fig, axs = plt.subplots()

sns.histplot(data=distances, bins="sqrt", ax=axs)
axs.set_xlabel("CO2 MSE Time difference (ms)")

In [ ]:
fig, axs = plt.subplots()

sns.histplot(data=distances, bins="sqrt", ax=axs)
axs.set_xlabel("ECE MSE Time difference (ms)")

# How to deal with missing channels?

## Ideas

- Only use channels that are always available across different discharges?
- Channel interpolation?
- ...

# Preprocessing

## Ideas

- All data have different physical units, and we need to get rid of them
- We should normalize the inputs before the ML task:
    - Standardization?
    - Min-Max normalization?
    - Robust methods like median- and percentile-based normalization?
    - Advanced methods for making skew distribution more similar to normal distributions?
- Inside models, we can use batch normalization

In [ ]:
fig, axs = plt.subplots()

sns.histplot(data=co2_samples[:, 0], bins="sqrt", ax=axs)
sns.histplot(data=co2_samples[:, 1], bins="sqrt", ax=axs)
sns.histplot(data=co2_samples[:, 2], bins="sqrt", ax=axs)
sns.histplot(data=co2_samples[:, 3], bins="sqrt", ax=axs)

In [ ]:
fig, axs = plt.subplots()

sns.histplot(data=X_trf[:, 0], bins="sqrt", ax=axs)
sns.histplot(data=X_trf[:, 1], bins="sqrt", ax=axs)
sns.histplot(data=X_trf[:, 2], bins="sqrt", ax=axs)
sns.histplot(data=X_trf[:, 3], bins="sqrt", ax=axs)

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_size, output_size, hidden_sizes):
        super(MLP, self).__init__()

        self.input_size = input_size
        self.output_size = output_size
        self.hidden_sizes = hidden_sizes

        self.layers = nn.ModuleList()

        # Add input layer
        self.layers.append(nn.Linear(input_size, hidden_sizes[0]))

        # Add hidden layers
        for i in range(len(hidden_sizes) - 1):
            self.layers.append(nn.Linear(hidden_sizes[i], hidden_sizes[i + 1]))

        # Add output layer
        self.layers.append(nn.Linear(hidden_sizes[-1], output_size))

    def forward(self, x):
        for layer in self.layers:
            x = torch.relu(layer(x))
        return x


# Example usage
input_size = 5  # co2
output_size = 1  # mse
hidden_sizes = [20, 30, 40]

model = MLP(input_size, output_size, hidden_sizes)
print(model)

Will be using valid padding (so start from before to after window)

In [ ]:
class VariableWindowDataset(Dataset):
    def __init__(self, data, labels, window_size):
        self.data = data
        self.labels = labels
        self.window_size = window_size

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        x = self.data[index]
        y = self.labels[index]

        start_index = index
        end_index = start_index + self.window_size
        x = x[start_index:end_index]

        return x, y


# Example usage
data = [...]  # Your data
labels = [...]  # Your labels
window_size = 10

dataset = VariableWindowDataset(data, labels, window_size)

# Create a dataloader
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Evaluation

# Potential applications?!